In [3]:
import pandas as pd 
import numpy as np 
import nltk
import gensim
import re 

In [22]:
from nltk.corpus import stopwords
nltk.download('stopwords')
from bs4 import BeautifulSoup

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
data=pd.read_csv('all_kindle_review.csv')

In [5]:
data

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000
...,...,...,...,...,...,...,...,...,...,...,...
11995,11995,2183,B001DUGORO,"[0, 0]",4,Valentine cupid is a vampire- Jena and Ian ano...,"02 28, 2014",A1OKS5Q1HD8WQC,lisa jon jung,jena,1393545600
11996,11996,6272,B002JCSFSQ,"[2, 2]",5,I have read all seven books in this series. Ap...,"05 16, 2011",AQRSPXLNEQAMA,TerryLP,Peacekeepers Series,1305504000
11997,11997,12483,B0035N1V7K,"[0, 1]",3,This book really just wasn't my cuppa. The si...,"07 26, 2013",A2T5QLT5VXOJAK,hwilson,a little creepy,1374796800
11998,11998,3640,B001W1XT40,"[1, 2]",1,"tried to use it to charge my kindle, it didn't...","09 17, 2013",A28MHD2DDY6DXB,"Allison A. Slater ""Gryphon50""",didn't work,1379376000


In [25]:
df=data[['reviewText','rating']]

In [26]:
df

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4
...,...,...
11995,Valentine cupid is a vampire- Jena and Ian ano...,4
11996,I have read all seven books in this series. Ap...,5
11997,This book really just wasn't my cuppa. The si...,3
11998,"tried to use it to charge my kindle, it didn't...",1


In [27]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [28]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [29]:
df['rating']=df['rating'].apply(lambda x :0 if x<3 else 1)

C:\Users\HP\AppData\Local\Temp\ipykernel_10424\2668202038.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['rating']=df['rating'].apply(lambda x :0 if x<3 else 1)


In [30]:
df['rating']

0        1
1        1
2        1
3        1
4        1
        ..
11995    1
11996    1
11997    1
11998    0
11999    1
Name: rating, Length: 12000, dtype: int64

In [31]:
df['rating'].shape

(12000,)

In [33]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

## Preprocessing

In [34]:
df['reviewText']=df['reviewText'].str.lower()

C:\Users\HP\AppData\Local\Temp\ipykernel_10424\2784960931.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].str.lower()


In [35]:
df['reviewText']

0        jace rankin may be short, but he's nothing to ...
1        great short read.  i didn't want to put it dow...
2        i'll start by saying this is the first of four...
3        aggie is angela lansbury who carries pocketboo...
4        i did not expect this type of book to be in li...
                               ...                        
11995    valentine cupid is a vampire- jena and ian ano...
11996    i have read all seven books in this series. ap...
11997    this book really just wasn't my cuppa.  the si...
11998    tried to use it to charge my kindle, it didn't...
11999    taking instruction is a look into the often hi...
Name: reviewText, Length: 12000, dtype: object

In [36]:
## Removing special characters
df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))
## Remove the stopswords
df['reviewText']=df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
## Remove url 
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove html tags
df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

C:\Users\HP\AppData\Local\Temp\ipykernel_10424\3736800740.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))
C:\Users\HP\AppData\Local\Temp\ipykernel_10424\3736800740.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
C:\Users\HP\AppData\Local\Temp\ipykernel_10424\3736800740.py:6: SettingWithCopyWarning:

In [39]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [41]:
from nltk.stem import WordNetLemmatizer
Lemmatizer=WordNetLemmatizer()

In [42]:
df['reviewText']=df['reviewText'].apply(lambda x:" ".join([Lemmatizer.lemmatize(word) for word in x.split()]))

C:\Users\HP\AppData\Local\Temp\ipykernel_10424\2097318463.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['reviewText']=df['reviewText'].apply(lambda x:" ".join([Lemmatizer.lemmatize(word) for word in x.split()]))


In [44]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [45]:
df['reviewText'].shape

(12000,)

In [46]:
df.shape

(12000, 2)

In [47]:
from sklearn.model_selection import train_test_split

In [48]:
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.20,
                                               random_state=42)

In [49]:
X_train

9182     looking forward book came double space every p...
11091    already owned book spouse forgot already part ...
6428     cool forgot request rate came make mine unreli...
288      short short story basically scene party one ni...
2626     secret service agent secrests even longer serv...
                               ...                        
11964    downloaded book reading review usually reading...
5191     far one hottest book ive ever gotten hand ondo...
5390     even though book free reservation based majori...
860      little mushy 34must take care woman folk34 cha...
7270     book good good set charaterswith background le...
Name: reviewText, Length: 9600, dtype: object

In [50]:
X_train.shape

(9600,)

In [51]:
X_test

1935         really great read wish would hope find author
6494     nope tried cant read take greatest delight del...
1720     story line drug like book much mystery fan wou...
9120     read several angel book one work didnt really ...
360      possibly worst book ever read beginning positi...
                               ...                        
1195     enjoyed read think fan humorous must err use l...
11877    pleasantly surprised book enjoyed m dubois tol...
5421     love best friend since 15 year old 30 he servi...
3855     fascinating book enough twist turn keep readin...
4414     plot noted publisher blurb publisher make fun ...
Name: reviewText, Length: 2400, dtype: object

In [52]:
X_test.shape

(2400,)

In [53]:
y_train

9182     1
11091    0
6428     1
288      0
2626     1
        ..
11964    0
5191     1
5390     0
860      1
7270     1
Name: rating, Length: 9600, dtype: int64

In [54]:
y_train.shape

(9600,)

In [55]:
y_test.shape

(2400,)

In [56]:
y_test

1935     1
6494     0
1720     0
9120     0
360      0
        ..
1195     1
11877    1
5421     1
3855     1
4414     1
Name: rating, Length: 2400, dtype: int64

In [57]:
X_train = X_train.apply(lambda x: x.split())
X_test = X_test.apply(lambda x: x.split())

In [58]:
X_train

9182     [looking, forward, book, came, double, space, ...
11091    [already, owned, book, spouse, forgot, already...
6428     [cool, forgot, request, rate, came, make, mine...
288      [short, short, story, basically, scene, party,...
2626     [secret, service, agent, secrests, even, longe...
                               ...                        
11964    [downloaded, book, reading, review, usually, r...
5191     [far, one, hottest, book, ive, ever, gotten, h...
5390     [even, though, book, free, reservation, based,...
860      [little, mushy, 34must, take, care, woman, fol...
7270     [book, good, good, set, charaterswith, backgro...
Name: reviewText, Length: 9600, dtype: object

In [59]:
X_test

1935     [really, great, read, wish, would, hope, find,...
6494     [nope, tried, cant, read, take, greatest, deli...
1720     [story, line, drug, like, book, much, mystery,...
9120     [read, several, angel, book, one, work, didnt,...
360      [possibly, worst, book, ever, read, beginning,...
                               ...                        
1195     [enjoyed, read, think, fan, humorous, must, er...
11877    [pleasantly, surprised, book, enjoyed, m, dubo...
5421     [love, best, friend, since, 15, year, old, 30,...
3855     [fascinating, book, enough, twist, turn, keep,...
4414     [plot, noted, publisher, blurb, publisher, mak...
Name: reviewText, Length: 2400, dtype: object

In [61]:
model=gensim.models.Word2Vec(
    sentences=X_train,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [65]:
def avg_word2vec(doc):
    # remove out-of-vocabulary words
    #sent = [word for word in doc if word in model.wv.index_to_key]
    #print(sent)
    
    return np.mean([model.wv[word] for word in doc if word in model.wv.index_to_key],axis=0)
                #or [np.zeros(len(model.wv.index_to_key))], axis=0)

In [66]:
from tqdm import tqdm

In [67]:
X_train_vec = np.array([avg_word2vec(sentence) for sentence in X_train])
X_test_vec = np.array([avg_word2vec(sentence) for sentence in X_test])

In [69]:
X_train_vec.shape

(9600, 100)

In [70]:
y_train.shape

(9600,)

In [71]:
X_test_vec

array([[-0.17000562,  0.41852534, -0.07475328, ..., -0.28599164,
         0.05789103, -0.1371515 ],
       [ 0.01560465,  0.22027199,  0.04992092, ..., -0.57493347,
         0.15046424, -0.04508229],
       [-0.19935079,  0.41197714,  0.23266195, ..., -0.2742214 ,
         0.00218694, -0.16270544],
       ...,
       [-0.08283205,  0.37837103, -0.10697449, ..., -0.56606716,
         0.2464505 ,  0.02046153],
       [-0.19768566,  0.34049037,  0.04224222, ..., -0.46369565,
        -0.02824758, -0.0398545 ],
       [-0.16307993,  0.38522908,  0.07058919, ..., -0.36092   ,
         0.0750994 , -0.12341969]], dtype=float32)

In [72]:
X_test.shape

(2400,)

In [73]:
y_test.shape

(2400,)

In [74]:
from sklearn.naive_bayes import MultinomialNB

In [75]:
from sklearn.ensemble import RandomForestClassifier

In [76]:
classifier=RandomForestClassifier()

In [77]:
model=classifier.fit(X_train_vec,y_train)

In [78]:
y_pred=model.predict(X_test_vec)

In [79]:
y_pred

array([1, 0, 1, ..., 1, 1, 1])

In [81]:
y_pred.shape,y_test.shape

((2400,), (2400,))

In [82]:
from sklearn.metrics import classification_report

In [83]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.68      0.55      0.61       803
           1       0.79      0.87      0.83      1597

    accuracy                           0.76      2400
   macro avg       0.74      0.71      0.72      2400
weighted avg       0.76      0.76      0.76      2400



In [88]:
from sklearn.linear_model import LogisticRegression

In [90]:
model2=LogisticRegression().fit(X_train_vec,y_train)

In [92]:
y_pred2=model2.predict(X_test_vec)

In [93]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.68      0.55      0.61       803
           1       0.79      0.87      0.83      1597

    accuracy                           0.76      2400
   macro avg       0.74      0.71      0.72      2400
weighted avg       0.76      0.76      0.76      2400

